In [1]:

from sklearn.datasets import make_classification
import pandas as pd

# Generate an imbalanced binary classification dataset
X, y = make_classification(
    n_samples=2000,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    weights=[0.95, 0.05],
    flip_y=0,
    random_state=42
)

# Convert to a structured DataFrame
feature_names = [f"Feature_{i}" for i in range(1, 13)]

X = pd.DataFrame(X, columns=feature_names)
y = pd.Series(y, name="Outcome")

# Examine class distribution
class_counts = y.value_counts().sort_index()
class_percentages = y.value_counts(
    normalize=True
).sort_index() * 100

print("DATASET SHAPE:", X.shape)
print("\nCLASS DISTRIBUTION")
print("-" * 40)

for cls in class_counts.index:
    print(
        f"Class {cls}: {class_counts[cls]} observations "
        f"({class_percentages[cls]:.1f}%)"
    )


DATASET SHAPE: (2000, 12)

CLASS DISTRIBUTION
----------------------------------------
Class 0: 1900 observations (95.0%)
Class 1: 100 observations (5.0%)


In [2]:

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Bar(
    x=["Class 0 — Majority", "Class 1 — Minority"],
    y=[1900, 100],
    marker_color=["#238B83", "#D97757"],
    text=["1,900 (95%)", "100 (5%)"],
    textposition="outside",
    cliponaxis=False,
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Observations: %{y:,}<extra></extra>"
    )
))

fig.update_layout(
    title=dict(
        text="Class Distribution — Imbalanced Dataset",
        x=0.02,
        y=0.96,
        xanchor="left",
        yanchor="top",
        font=dict(size=19)
    ),
    xaxis_title="Outcome Class",
    yaxis_title="Number of Observations",
    template="plotly_white",
    width=850,
    height=550,
    margin=dict(l=90, r=50, t=100, b=100),
    showlegend=False,
    font=dict(size=12)
)

fig.update_yaxes(
    range=[0, 2100],
    title_standoff=20,
    automargin=True
)

fig.update_xaxes(
    title_standoff=20,
    automargin=True
)

fig.show()


In [3]:

from sklearn.model_selection import train_test_split

# Split the dataset while preserving class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("TRAINING DATA")
print("-" * 40)
print("Features:", X_train.shape)
print("Class distribution:")
print(y_train.value_counts().sort_index())

print("\nTESTING DATA")
print("-" * 40)
print("Features:", X_test.shape)
print("Class distribution:")
print(y_test.value_counts().sort_index())


TRAINING DATA
----------------------------------------
Features: (1600, 12)
Class distribution:
Outcome
0    1520
1      80
Name: count, dtype: int64

TESTING DATA
----------------------------------------
Features: (400, 12)
Class distribution:
Outcome
0    380
1     20
Name: count, dtype: int64


In [4]:

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Build baseline model without SMOTE
baseline_model = Pipeline([
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train on original imbalanced training data
baseline_model.fit(X_train, y_train)

# Predict test classifications
baseline_pred = baseline_model.predict(X_test)

# Predict minority-class probabilities
baseline_prob = baseline_model.predict_proba(X_test)[:, 1]

# Evaluate baseline performance
print("BASELINE LOGISTIC REGRESSION — WITHOUT SMOTE")
print("-" * 55)

print(f"Accuracy:          {accuracy_score(y_test, baseline_pred):.2%}")
print(f"Precision:         {precision_score(y_test, baseline_pred, zero_division=0):.2%}")
print(f"Recall:            {recall_score(y_test, baseline_pred):.2%}")
print(f"F1 Score:          {f1_score(y_test, baseline_pred):.3f}")
print(f"Average Precision: {average_precision_score(y_test, baseline_prob):.3f}")

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, baseline_pred))

print("\nCLASSIFICATION REPORT")
print(classification_report(y_test, baseline_pred, zero_division=0))


BASELINE LOGISTIC REGRESSION — WITHOUT SMOTE
-------------------------------------------------------
Accuracy:          95.75%
Precision:         66.67%
Recall:            30.00%
F1 Score:          0.414
Average Precision: 0.510

CONFUSION MATRIX
[[377   3]
 [ 14   6]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.96      0.99      0.98       380
           1       0.67      0.30      0.41        20

    accuracy                           0.96       400
   macro avg       0.82      0.65      0.70       400
weighted avg       0.95      0.96      0.95       400



In [5]:

%pip install imbalanced-learn


In [6]:

import imblearn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

print("imbalanced-learn installed successfully!")
print("Version:", imblearn.__version__)
print("SMOTE and ImbPipeline are ready.")


imbalanced-learn installed successfully!
Version: 0.14.2
SMOTE and ImbPipeline are ready.


In [7]:

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Build a pipeline with SMOTE
smote_model = ImbPipeline([
    ("scaler", StandardScaler()),
    ("smote", SMOTE(random_state=42)),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Fit on training data ONLY
smote_model.fit(X_train, y_train)

# Predict on the original, untouched test data
smote_pred = smote_model.predict(X_test)
smote_prob = smote_model.predict_proba(X_test)[:, 1]

# Evaluate the model
print("LOGISTIC REGRESSION — WITH SMOTE")
print("-" * 55)

print(f"Accuracy:          {accuracy_score(y_test, smote_pred):.2%}")
print(f"Precision:         {precision_score(y_test, smote_pred, zero_division=0):.2%}")
print(f"Recall:            {recall_score(y_test, smote_pred):.2%}")
print(f"F1 Score:          {f1_score(y_test, smote_pred):.3f}")
print(f"Average Precision: {average_precision_score(y_test, smote_prob):.3f}")

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, smote_pred))

print("\nCLASSIFICATION REPORT")
print(classification_report(y_test, smote_pred, zero_division=0))


LOGISTIC REGRESSION — WITH SMOTE
-------------------------------------------------------
Accuracy:          86.00%
Precision:         22.73%
Recall:            75.00%
F1 Score:          0.349
Average Precision: 0.508

CONFUSION MATRIX
[[329  51]
 [  5  15]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.99      0.87      0.92       380
           1       0.23      0.75      0.35        20

    accuracy                           0.86       400
   macro avg       0.61      0.81      0.64       400
weighted avg       0.95      0.86      0.89       400



In [8]:

import pandas as pd
import plotly.graph_objects as go

comparison_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score",
        "Average Precision"
    ],
    "Without SMOTE": [
        0.9575, 0.6667, 0.30, 0.414, 0.510
    ],
    "With SMOTE": [
        0.86, 0.2273, 0.75, 0.349, 0.508
    ]
})

fig = go.Figure()

for name, color in [
    ("Without SMOTE", "#64748B"),
    ("With SMOTE", "#238B83")
]:
    fig.add_trace(go.Bar(
        name=name,
        x=comparison_df["Metric"],
        y=comparison_df[name],
        marker_color=color,
        text=comparison_df[name].map(lambda x: f"{x:.1%}"),
        textposition="outside",
        cliponaxis=False,
        hovertemplate=(
            "<b>%{x}</b><br>"
            f"{name}: " + "%{y:.2%}"
            "<extra></extra>"
        )
    ))

fig.update_layout(
    title=dict(
        text="Baseline vs SMOTE — Classification Performance",
        x=0.02,
        y=0.97,
        xanchor="left",
        yanchor="top",
        font=dict(size=19)
    ),
    barmode="group",
    template="plotly_white",
    width=1000,
    height=600,
    margin=dict(l=90, r=50, t=155, b=100),
    legend=dict(
        orientation="h",
        x=0,
        y=1.12,
        xanchor="left",
        yanchor="bottom",
        font=dict(size=12)
    ),
    xaxis_title="Evaluation Metric",
    yaxis_title="Score",
    font=dict(size=12)
)

fig.update_yaxes(
    range=[0, 1.10],
    tickformat=".0%",
    title_standoff=20,
    automargin=True
)

fig.update_xaxes(
    title_standoff=20,
    automargin=True
)

fig.show()


In [9]:

from sklearn.model_selection import StratifiedKFold, cross_validate
import pandas as pd

# Create stratified 5-fold cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Metrics focused on minority-class detection
scoring = {
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1",
    "Average Precision": "average_precision"
}

# Evaluate both pipelines on training data only
results = {}

for name, pipeline in [
    ("Without SMOTE", baseline_model),
    ("With SMOTE", smote_model)
]:
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    results[name] = {
        metric: scores[f"test_{metric}"].mean()
        for metric in scoring
    }

# Create comparison table
cv_comparison_df = pd.DataFrame(results).T

print("5-FOLD CROSS-VALIDATION RESULTS")
print("-" * 65)
print(cv_comparison_df.round(3).to_string())


5-FOLD CROSS-VALIDATION RESULTS
-----------------------------------------------------------------
               Accuracy  Precision  Recall     F1  Average Precision
Without SMOTE     0.964      0.810   0.388  0.519              0.601
With SMOTE        0.871      0.254   0.788  0.383              0.568


In [10]:

import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Obtain out-of-fold probabilities from training data
cv_baseline_prob = cross_val_predict(
    baseline_model,
    X_train,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

# Examine several classification thresholds
threshold_results = []

for threshold in [0.10, 0.20, 0.30, 0.40, 0.50]:
    predictions = (cv_baseline_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_train, predictions, labels=[0, 1]
    ).ravel()

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(
            y_train, predictions, zero_division=0
        ),
        "Recall": recall_score(y_train, predictions),
        "F1": f1_score(y_train, predictions),
        "False Positives": fp,
        "False Negatives": fn
    })

threshold_df = pd.DataFrame(threshold_results)

print("BASELINE MODEL — CROSS-VALIDATED THRESHOLD COMPARISON")
print("-" * 75)
print(threshold_df.round(3).to_string(index=False))


BASELINE MODEL — CROSS-VALIDATED THRESHOLD COMPARISON
---------------------------------------------------------------------------
 Threshold  Precision  Recall    F1  False Positives  False Negatives
       0.1      0.320   0.712 0.442              121               23
       0.2      0.470   0.588 0.522               53               33
       0.3      0.619   0.488 0.545               24               41
       0.4      0.673   0.412 0.512               16               47
       0.5      0.795   0.388 0.521                8               49


In [11]:

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    average_precision_score
)

# Threshold selected from training-data CV
selected_threshold = 0.30

# Use the already fitted baseline model
threshold_test_pred = (
    baseline_prob >= selected_threshold
).astype(int)

print("BASELINE LOGISTIC REGRESSION — THRESHOLD 0.30")
print("-" * 60)

print(f"Accuracy:          {accuracy_score(y_test, threshold_test_pred):.2%}")
print(f"Precision:         {precision_score(y_test, threshold_test_pred, zero_division=0):.2%}")
print(f"Recall:            {recall_score(y_test, threshold_test_pred):.2%}")
print(f"F1 Score:          {f1_score(y_test, threshold_test_pred):.3f}")
print(f"Average Precision: {average_precision_score(y_test, baseline_prob):.3f}")

print("\nCONFUSION MATRIX")
print(confusion_matrix(y_test, threshold_test_pred))


BASELINE LOGISTIC REGRESSION — THRESHOLD 0.30
------------------------------------------------------------
Accuracy:          94.50%
Precision:         45.00%
Recall:            45.00%
F1 Score:          0.450
Average Precision: 0.510

CONFUSION MATRIX
[[369  11]
 [ 11   9]]


In [12]:

import plotly.graph_objects as go
from sklearn.metrics import precision_recall_curve, average_precision_score

fig = go.Figure()

models = [
    ("Without SMOTE", baseline_prob, "#64748B"),
    ("With SMOTE", smote_prob, "#238B83")
]

for name, probabilities, color in models:
    precision, recall, thresholds = precision_recall_curve(
        y_test, probabilities
    )

    ap = average_precision_score(y_test, probabilities)

    fig.add_trace(go.Scatter(
        x=recall,
        y=precision,
        mode="lines",
        name=f"{name} (AP={ap:.3f})",
        line=dict(color=color, width=3),
        hovertemplate=(
            "Recall: %{x:.2%}<br>"
            "Precision: %{y:.2%}"
            "<extra></extra>"
        )
    ))

# Reference precision for the minority-class prevalence
fig.add_hline(
    y=y_test.mean(),
    line_dash="dash",
    line_color="gray",
    annotation_text="Minority prevalence (5%)",
    annotation_position="bottom right"
)

fig.update_layout(
    title=dict(
        text="Precision–Recall Curves — Baseline vs SMOTE",
        x=0.02,
        y=0.97,
        xanchor="left",
        yanchor="top",
        font=dict(size=19)
    ),
    xaxis_title="Recall (Sensitivity)",
    yaxis_title="Precision",
    template="plotly_white",
    width=950,
    height=650,
    margin=dict(l=95, r=65, t=160, b=100),
    legend=dict(
        orientation="h",
        x=0,
        y=1.12,
        xanchor="left",
        yanchor="bottom",
        font=dict(size=12)
    ),
    font=dict(size=12)
)

fig.update_xaxes(
    range=[0, 1.02],
    tickformat=".0%",
    title_standoff=20,
    automargin=True
)

fig.update_yaxes(
    range=[0, 1.02],
    tickformat=".0%",
    title_standoff=20,
    automargin=True
)

fig.show()
